# Sprint 1.1 — LLM Baseline : Limites des LLM sans RAG

**Objectif** : Comparer 3 LLM open-source sur des questions logement et constater les hallucinations.

**Sujets couverts** :
- Architecture Transformer et mécanisme d'attention
- Principaux LLM disponibles (Llama, Mistral, Qwen)
- Forces et faiblesses des LLM nus
- Motivation pour le RAG

**Pré-requis** : Ollama installé + au moins un modèle chargé

In [ ]:
# Installation (si pas déjà fait)
# !pip install langchain langchain-community ollama

## 1. Setup — Connexion à Ollama

Ollama doit être lancé (`ollama serve`) avec les modèles téléchargés :
```bash
ollama pull llama3.1:8b
ollama pull mistral:7b
ollama pull qwen2.5:7b
```

In [ ]:
from langchain_community.chat_models import ChatOllama

# Les 3 modèles à comparer
models = {
    "Llama 3.1 8B": ChatOllama(model="llama3.1:8b", temperature=0.1),
    "Mistral 7B": ChatOllama(model="mistral:7b", temperature=0.1),
    "Qwen 2.5 7B": ChatOllama(model="qwen2.5:7b", temperature=0.1),
}

# Test de connexion
for name, llm in models.items():
    try:
        resp = llm.invoke("Dis 'OK' en un mot.")
        print(f"✓ {name} : {resp.content[:20]}")
    except Exception as e:
        print(f"✗ {name} : {e}")

## 2. Model Showdown — 5 questions logement

Nous allons poser les mêmes 5 questions à chaque modèle.
Objectif : identifier les hallucinations et les limites.

In [ ]:
QUESTIONS = [
    "À quelle température régler une chaudière Viessmann Vitodens la nuit ?",
    "Quelle est la durée légale d'un préavis de départ en zone tendue ?",
    "Comment dégivrer un réfrigérateur Bosch modèle KGN56XLEA ?",
    "Que signifie l'erreur F4 sur une chaudière à condensation ?",
    "Quelle est la consommation électrique normale d'un appartement de 68m2 en classe D ?",
]

import pandas as pd

results = []
for question in QUESTIONS:
    row = {"Question": question[:60] + "..."}
    for name, llm in models.items():
        try:
            resp = llm.invoke(question)
            row[name] = resp.content[:200] + "..." if len(resp.content) > 200 else resp.content
        except Exception as e:
            row[name] = f"ERREUR: {e}"
    results.append(row)

df_results = pd.DataFrame(results)
print(df_results.to_string())

## 3. Analyse des hallucinations

**Questions clés pour l'analyse :**
- Les modèles ont-ils donné des températures précises pour la chaudière ? (ils ne peuvent pas savoir — notice non fournie)
- Ont-ils correctement indiqué le préavis de 1 mois en zone tendue ?
- Ont-ils inventé des spécifications pour le frigo Bosch ?
- Connaissaient-ils le code F4 ?

**Conclusion attendue :** Les LLM nus hallucinent sur les informations spécifiques au logement.

In [ ]:
# Tableau comparatif avec notes de confiance (à remplir manuellement)
evaluation = pd.DataFrame({
    "Question": [q[:40]+"..." for q in QUESTIONS],
    "Llama 3.1 8B (précision)": ["/5", "/5", "/5", "/5", "/5"],
    "Mistral 7B (précision)": ["/5", "/5", "/5", "/5", "/5"],
    "Qwen 2.5 7B (précision)": ["/5", "/5", "/5", "/5", "/5"],
})
print(evaluation)
print("\n→ Remplissez ce tableau après avoir analysé les réponses.")

## 4. Conclusion — Pourquoi le RAG ?

**Problèmes identifiés :**
1. **Hallucinations** : Les LLM inventent des informations spécifiques qu'ils ne peuvent pas connaître
2. **Données périmées** : La cut-off date du modèle ne reflète pas votre situation actuelle
3. **Pas de personnalisation** : Le modèle ne connaît pas VOTRE chaudière, VOTRE bail
4. **Pas de sources** : Impossible de vérifier d'où vient l'information

**Solution : RAG (Retrieval-Augmented Generation)**
→ Fournir au LLM le contexte documentaire pertinant AVANT de générer la réponse.

**Sprint 1.2** : Nous allons implémenter ce pipeline RAG !

## Bonus — Théorie : Architecture Transformer

### Mécanisme d'attention (simplifié)
```
Input tokens → Embedding → Q, K, V matrices → Attention scores → Output
```

- **Query (Q)** : "Que cherche ce token ?"
- **Key (K)** : "Quelle information contient ce token ?"
- **Value (V)** : "Quelle est la valeur de ce token ?"
- **Attention** = softmax(QK^T / √d_k) × V

### Familles de modèles
| Modèle | Paramètres | Contexte | Licence |
|---|---|---|---|
| Llama 3.1 8B | 8B | 128K | Meta Llama |
| Mistral 7B | 7B | 32K | Apache 2.0 |
| Qwen 2.5 7B | 7B | 128K | Qwen |
| Phi-3 mini | 3.8B | 128K | MIT |

### Écosystème Python
- **HuggingFace Transformers** : inférence locale, fine-tuning
- **LangChain** : orchestration, agents, RAG
- **Ollama** : serving simplifié local
- **FAISS / ChromaDB** : bases vectorielles